# ETTh эксперименты: Informer + Optuna

В этом ноутбуке запускаются эксперименты **Informer + Optuna** на датасетах `ETTh1` и `ETTh2`.

Основные шаги:

1. Настройка окружения и конфигурации экспериментов.
2. Для каждого значения `MAX_ROWS` создаётся усечённый CSV-файл.
3. Запускается Informer + Optuna для каждой комбинации датасет × `MAX_ROWS`.
4. Загружаются сохранённые предсказания Informer и строятся графики.
5. Формируется сводная таблица метрик по всем запускам.

Внешний код Informer предоставляет предсказания только для отложенной выборки, поэтому:
- для train-части строится график фактических значений;
- для валидационной/тестовой части строится график фактических и предсказанных значений.

In [1]:
from __future__ import annotations

import logging
import os
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Dict
from typing import List
from typing import Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

CURRENT_DIR: Path = Path.cwd().resolve()
REPO_ROOT_CANDIDATES: List[Path] = [
    CURRENT_DIR,
    CURRENT_DIR.parent,
]

REPO_ROOT: Path | None = None
for candidate in REPO_ROOT_CANDIDATES:
    if (candidate / 'src' / 'edlm_search').is_dir():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise RuntimeError(
            f'Cannot locate project root with "src/edlm_search" directory from "{CURRENT_DIR}".'
    )

SRC_DIR: Path = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from edlm_search.experiments import ExperimentResult, run_informer_optuna_etth_experiment
from edlm_search.experiments.datasets import load_ett_csv_dataset

/Users/roman/Projects/PycharmProjects/ITMO/edlm_search/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:


ENV_PATH: Path = REPO_ROOT / '.env'
if ENV_PATH.is_file():
    load_dotenv(dotenv_path=ENV_PATH)

ETT_DATA_DIR: Path = SRC_DIR / 'ETDataset' / 'ETT-small'
ETTH1_PATH: Path = ETT_DATA_DIR / 'ETTh1.csv'
ETTH2_PATH: Path = ETT_DATA_DIR / 'ETTh2.csv'

DATASET_NAMES: List[str] = ['ETTh1', 'ETTh2']

# MAX_ROWS_VALUES: List[int] = [5000, 10000, 20000]
MAX_ROWS_VALUES: List[int] = [5000]

TRAIN_RATIO: float = float(os.getenv('TRAIN_RATIO', '0.8'))
TARGET_COLUMN: str = os.getenv('TARGET_COLUMN', 'OT')

INFORMER_TIMEOUT_SECONDS: int = int(os.getenv('INFORMER_TIMEOUT_SECONDS', '3600'))
INFORMER_N_TRIALS: int = int(os.getenv('INFORMER_N_TRIALS', '10'))
INFORMER_METRICS_ROOT: Path = (
        REPO_ROOT / os.getenv('INFORMER_METRICS_ROOT', 'artifacts/informer_optuna').strip()
).resolve()

INFORMER_DATASETS_ROOT: Path = (
        REPO_ROOT / 'artifacts' / 'informer_datasets'
).resolve()

PLOTS_MAX_POINTS: int = 500

LOG_LEVEL: int = logging.INFO
logging.basicConfig(
        level=LOG_LEVEL,
        format='%(asctime)s - %(levelname)s - %(name)s - %(message)s',
)

DEVICE_TYPE = os.getenv('DEVICE_TYPE', 'auto')

LOGGER = logging.getLogger('etth_informer_optuna_experiments')

INFORMER_METRICS_ROOT.mkdir(parents=True, exist_ok=True)
INFORMER_DATASETS_ROOT.mkdir(parents=True, exist_ok=True)

LOGGER.info(f'Repository root resolved to "{REPO_ROOT}".')
LOGGER.info(f'ETT data directory resolved to "{ETT_DATA_DIR}".')
LOGGER.info(f'Informer metrics root resolved to "{INFORMER_METRICS_ROOT}".')
LOGGER.info(f'Informer datasets root resolved to "{INFORMER_DATASETS_ROOT}".')
LOGGER.info(f'Informer Optuna experiments will use device_type="{DEVICE_TYPE}".')

2025-11-21 20:26:40,581 - INFO - etth_informer_optuna_experiments - Repository root resolved to "/Users/roman/Projects/PycharmProjects/ITMO/edlm_search".
2025-11-21 20:26:40,581 - INFO - etth_informer_optuna_experiments - ETT data directory resolved to "/Users/roman/Projects/PycharmProjects/ITMO/edlm_search/src/ETDataset/ETT-small".
2025-11-21 20:26:40,582 - INFO - etth_informer_optuna_experiments - Informer metrics root resolved to "/Users/roman/Projects/PycharmProjects/ITMO/edlm_search/artifacts/informer_optuna".
2025-11-21 20:26:40,582 - INFO - etth_informer_optuna_experiments - Informer datasets root resolved to "/Users/roman/Projects/PycharmProjects/ITMO/edlm_search/artifacts/informer_datasets".
2025-11-21 20:26:40,582 - INFO - etth_informer_optuna_experiments - Informer Optuna experiments will use device_type="mps".


## Конфигурация и структуры данных

В этом разделе определяются:

- описания датасетов;
- конфигурация экспериментов Informer + Optuna;
- служебные структуры для идентификации запусков и путей к данным.

In [3]:
@dataclass(frozen=True)
class DatasetConfig:
    """Dataset configuration for ETTh experiments."""

    name: str
    csv_path: Path
    train_ratio: float


@dataclass(frozen=True)
class InformerOptunaExperimentConfig:
    """Configuration for Informer + Optuna experiments."""

    timeout_seconds: int
    n_trials: int
    metrics_root_dir: Path
    device_type: str
    max_rows_values: List[int]
    plots_max_points: int


@dataclass(frozen=True)
class InformerRunKey:
    """Identifier of a single Informer experiment run."""

    dataset_name: str
    max_rows: int

## Вспомогательные функции и класс запуска экспериментов Informer

Далее определены функции для:

- подготовки усечённых CSV-файлов под разные `MAX_ROWS`;
- загрузки и разбиения датасетов;
- построения графиков для train и для предсказаний Informer.

Класс `InformerOptunaExperimentRunner` управляет запуском всех экспериментов и
формированием сводной таблицы метрик.

In [4]:
def prepare_trimmed_csv(
        original_csv_path: Path,
        dataset_name: str,
        max_rows: int,
        datasets_root: Path,
) -> Path:
    """
    Create a trimmed CSV file with at most `max_rows` rows for a dataset.

    Parameters
    ----------
    original_csv_path : Path
        Path to the original full CSV file.
    dataset_name : str
        Name of the dataset.
    max_rows : int
        Maximum number of rows to keep.
    datasets_root : Path
        Root directory where trimmed CSV files will be stored.

    Returns
    -------
    Path
        Path to the trimmed CSV file.
    """
    if max_rows <= 0:
        raise ValueError('max_rows must be a positive integer.')
    if not original_csv_path.is_file():
        raise FileNotFoundError(
                f'Original CSV for dataset "{dataset_name}" not found at "{original_csv_path}".'
        )

    dataset_dir = datasets_root / dataset_name
    dataset_dir.mkdir(parents=True, exist_ok=True)
    trimmed_csv_path = dataset_dir / f'{dataset_name}_max_rows_{max_rows}.csv'

    df = pd.read_csv(original_csv_path)
    if len(df) > max_rows:
        df = df.head(max_rows)
    df.to_csv(trimmed_csv_path, index=False)

    LOGGER.info(
            f'Trimmed CSV for dataset="{dataset_name}" with max_rows={max_rows} '
            f'written to "{trimmed_csv_path}".'
    )
    return trimmed_csv_path


def load_train_valid_for_visualization(
        csv_path: Path,
        max_rows: int,
        train_ratio: float,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Load and split ETTh dataset for visualization.

    Parameters
    ----------
    csv_path : Path
        Path to the CSV file.
    max_rows : int
        Maximum number of rows to load.
    train_ratio : float
        Train split ratio.

    Returns
    -------
    tuple[pd.DataFrame, pd.DataFrame]
        Split (train_df, valid_df).
    """
    train_df, valid_df = load_ett_csv_dataset(
            csv_path=str(csv_path),
            max_rows=max_rows,
            train_ratio=train_ratio,
    )
    return train_df, valid_df


def plot_time_series_single(
        x_values: np.ndarray,
        y_values: np.ndarray,
        title: str,
        label: str,
) -> None:
    """
    Plot a single time series.

    Parameters
    ----------
    x_values : np.ndarray
        X-axis values (for example, datetime index).
    y_values : np.ndarray
        Values to plot.
    title : str
        Title of the plot.
    label : str
        Label for the series.
    """
    if x_values.shape[0] != y_values.shape[0]:
        raise ValueError('x_values and y_values must have the same length for plotting.')

    plt.figure(figsize=(18, 4))
    plt.plot(x_values, y_values, label=label)
    plt.xlabel('time')
    plt.ylabel('value')
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.gcf().autofmt_xdate()
    plt.show()


def plot_true_vs_pred(
        x_values: np.ndarray,
        y_true: np.ndarray,
        y_pred: np.ndarray,
        title: str,
) -> None:
    """
    Plot ground truth and predictions for Informer evaluation subset.

    Parameters
    ----------
    x_values : np.ndarray
        X-axis values (for example, datetime index).
    y_true : np.ndarray
        Ground truth values.
    y_pred : np.ndarray
        Predicted values.
    title : str
        Title of the plot.
    """
    if y_true.shape != y_pred.shape:
        raise ValueError('Shapes of y_true and y_pred must match for plotting.')
    if x_values.shape[0] != y_true.shape[0]:
        raise ValueError('x_values and y_true must have the same length for plotting.')

    plt.figure(figsize=(18, 4))
    plt.plot(x_values, y_true, label='y_true')
    plt.plot(x_values, y_pred, label='y_pred')
    plt.xlabel('time')
    plt.ylabel('value')
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.gcf().autofmt_xdate()
    plt.show()


def build_informer_base_extra_args(device_type: str) -> List[str] | None:
    """
    Build base extra arguments list for Informer CLI based on device type string.

    Parameters
    ----------
    device_type : str
        Device type value from configuration or environment. Supported values:
        "auto", "cpu", "cuda", "mps".

    Returns
    -------
    list[str] | None
        List of extra CLI arguments for Informer or None if no extra arguments
        should be added (for example, when device_type == "auto").
    """
    if device_type == '':
        raise ValueError('device_type must not be empty.')
    normalized = device_type.strip().lower()
    if normalized == 'auto':
        return None
    if normalized in ('cpu', 'cuda', 'mps'):
        return ['--device_type', normalized]
    raise ValueError(
            f'Unsupported DEVICE_TYPE value "{device_type}". '
            f'Expected one of ["auto", "cpu", "cuda", "mps"].'
    )


def build_predictions_csv_path(
        metrics_root_dir: Path,
        dataset_name: str,
        max_rows: int,
) -> Path:
    """
    Build path to a predictions CSV file for a given dataset and max_rows.

    Parameters
    ----------
    metrics_root_dir : Path
        Root directory where Informer metrics and artifacts are stored.
    dataset_name : str
        Name of the dataset.
    max_rows : int
        Maximum number of rows used in the experiment.

    Returns
    -------
    Path
        Path to the predictions CSV file.
    """
    if dataset_name == '':
        raise ValueError('dataset_name must not be empty.')
    if max_rows <= 0:
        raise ValueError('max_rows must be a positive integer.')
    dataset_dir = metrics_root_dir / dataset_name / f'max_rows_{max_rows}'
    predictions_csv_path = dataset_dir / 'predictions.csv'
    return predictions_csv_path


class InformerOptunaExperimentRunner:
    """Manage Informer + Optuna experiments for multiple datasets and max_rows values."""

    def __init__(
            self,
            dataset_configs: List[DatasetConfig],
            experiment_config: InformerOptunaExperimentConfig,
    ) -> None:
        if not dataset_configs:
            raise ValueError('dataset_configs must not be empty.')
        if experiment_config.device_type == '':
            raise ValueError('experiment_config.device_type must not be empty.')
        self._dataset_configs = list(dataset_configs)
        self._experiment_config = experiment_config

    def run_all(self) -> Dict[InformerRunKey, ExperimentResult]:
        """Run experiments for all datasets and all max_rows values."""
        results: Dict[InformerRunKey, ExperimentResult] = {}
        for dataset_config in self._dataset_configs:
            for max_rows in self._experiment_config.max_rows_values:
                run_key = InformerRunKey(dataset_name=dataset_config.name, max_rows=max_rows)
                LOGGER.info(
                        f'Starting Informer+Optuna run for dataset="{run_key.dataset_name}", '
                        f'max_rows={run_key.max_rows}.'
                )
                result = self._run_single(dataset_config, max_rows)
                results[run_key] = result
                mse_value = float(result.metrics.get('mse', float('nan')))
                LOGGER.info(
                        f'Finished Informer+Optuna run for dataset="{run_key.dataset_name}", '
                        f'max_rows={run_key.max_rows}, mse={mse_value}.'
                )
        return results

    def _run_single(
            self,
            dataset_config: DatasetConfig,
            max_rows: int,
    ) -> ExperimentResult:
        """Run a single Informer + Optuna experiment for a dataset and max_rows."""
        if max_rows <= 0:
            raise ValueError('max_rows must be a positive integer.')
        if not dataset_config.csv_path.is_file():
            raise FileNotFoundError(
                    f'Dataset "{dataset_config.name}" CSV not found at "{dataset_config.csv_path}".'
            )

        trimmed_csv = prepare_trimmed_csv(
                original_csv_path=dataset_config.csv_path,
                dataset_name=dataset_config.name,
                max_rows=max_rows,
                datasets_root=INFORMER_DATASETS_ROOT,
        )

        metrics_root_for_dataset = self._experiment_config.metrics_root_dir / dataset_config.name / f'max_rows_{max_rows}'
        metrics_root_for_dataset.mkdir(parents=True, exist_ok=True)

        base_extra_args = build_informer_base_extra_args(
                device_type=self._experiment_config.device_type,
        )

        result = run_informer_optuna_etth_experiment(
                dataset_name=dataset_config.name,
                csv_path=str(trimmed_csv),
                informer_script_path=str(SRC_DIR / 'Informer2020' / 'informer_experiment_wrapper.py'),
                metrics_root_dir=str(metrics_root_for_dataset),
                base_extra_args=base_extra_args,
                timeout_seconds=self._experiment_config.timeout_seconds,
                model_name=f'informer-optuna-{dataset_config.name.lower()}',
                n_trials=self._experiment_config.n_trials,
        )
        return result

    @staticmethod
    def build_metrics_dataframe(
            results: Dict[InformerRunKey, ExperimentResult],
            primary_metric: str,
    ) -> pd.DataFrame:
        """Convert experiment results into a flat metrics DataFrame."""
        rows: List[Dict[str, float | int | str]] = []
        for run_key, result in results.items():
            metrics = result.metrics
            row: Dict[str, float | int | str] = {
                'dataset': run_key.dataset_name,
                'max_rows': run_key.max_rows,
            }
            for metric_name, metric_value in metrics.items():
                row[metric_name] = float(metric_value)
            if primary_metric not in row:
                row[primary_metric] = float('nan')
            rows.append(row)
        if not rows:
            return pd.DataFrame()
        df = pd.DataFrame(rows)
        df.sort_values(by=['dataset', 'max_rows'], inplace=True)
        df.reset_index(drop=True, inplace=True)
        return df

## Запуск экспериментов Informer + Optuna

В этом разделе:

1. Создаются конфигурации датасетов и эксперимента.
2. Запускаются все эксперименты Informer + Optuna для комбинаций датасет × `MAX_ROWS`.
3. Формируется таблица метрик по всем запускам.

In [5]:
dataset_configs: List[DatasetConfig] = [
    DatasetConfig(name='ETTh1', csv_path=ETTH1_PATH, train_ratio=TRAIN_RATIO),
    DatasetConfig(name='ETTh2', csv_path=ETTH2_PATH, train_ratio=TRAIN_RATIO),
]

informer_experiment_config = InformerOptunaExperimentConfig(
        timeout_seconds=INFORMER_TIMEOUT_SECONDS,
        n_trials=INFORMER_N_TRIALS,
        metrics_root_dir=INFORMER_METRICS_ROOT,
        device_type=DEVICE_TYPE,
        max_rows_values=MAX_ROWS_VALUES,
        plots_max_points=PLOTS_MAX_POINTS,
)

informer_runner = InformerOptunaExperimentRunner(
        dataset_configs=dataset_configs,
        experiment_config=informer_experiment_config,
)

RUN_INFORMER_OPTUNA_EXPERIMENTS: bool = True

if RUN_INFORMER_OPTUNA_EXPERIMENTS:
    informer_results_by_run_key: Dict[InformerRunKey, ExperimentResult] = informer_runner.run_all()
    primary_metric_name: str = 'mse'
    informer_metrics_df = InformerOptunaExperimentRunner.build_metrics_dataframe(
            results=informer_results_by_run_key,
            primary_metric=primary_metric_name,
    )
    LOGGER.info(
            f'Informer+Optuna experiments finished, metrics DataFrame shape={informer_metrics_df.shape}.'
    )
else:
    informer_results_by_run_key = {}
    informer_metrics_df = pd.DataFrame()
    LOGGER.info(
            f'Informer+Optuna experiments skipped, RUN_INFORMER_OPTUNA_EXPERIMENTS={RUN_INFORMER_OPTUNA_EXPERIMENTS}.'
    )

2025-11-21 20:26:40,636 - INFO - etth_informer_optuna_experiments - Starting Informer+Optuna run for dataset="ETTh1", max_rows=5000.
2025-11-21 20:26:40,679 - INFO - etth_informer_optuna_experiments - Trimmed CSV for dataset="ETTh1" with max_rows=5000 written to "/Users/roman/Projects/PycharmProjects/ITMO/edlm_search/artifacts/informer_datasets/ETTh1/ETTh1_max_rows_5000.csv".
2025-11-21 20:26:40,680 - INFO - edlm_search.experiments.experiment_api - Informer search space for dataset "ETTh1": d_model=[128, 192, 256, 320, 384, 448, 512], n_heads=[2, 4, 8], e_layers=[1, 2, 3], d_layers=[1, 2, 3], factor=[1, 3, 5], dropout=[0.05, 0.1, 0.2, 0.3], learning_rate=[1e-05, 3e-05, 0.0001, 0.0003, 0.001], batch_size=[16, 32, 64], epochs=[5, 10, 20].
2025-11-21 20:26:40,680 - INFO - edlm_search.experiments.experiment_api - Запуск Informer+Optuna для датасета "ETTh1" с 1 испытаниями.
[I 2025-11-21 20:26:40,680] A new study created in memory with name: informer_etth_mse_ETTh1
2025-11-21 20:26:40,681 -

KeyboardInterrupt: 

In [ ]:
informer_metrics_df

## Визуализация результатов Informer

На этом шаге для каждой комбинации датасет × `MAX_ROWS`:

1. Строится график фактических значений `TARGET_COLUMN` на train-части выборки.
2. Загружается CSV с предсказаниями Informer и строится график фактических и предсказанных значений
   для отложенной выборки (валидация/тест).

In [ ]:
for dataset_name in DATASET_NAMES:
    for max_rows in MAX_ROWS_VALUES:
        LOGGER.info(
                f'Starting visualization for dataset="{dataset_name}", max_rows={max_rows}.'
        )

        dataset_cfg = next((cfg for cfg in dataset_configs if cfg.name == dataset_name), None)
        if dataset_cfg is None:
            LOGGER.info(
                    f'Visualization skipped for dataset="{dataset_name}", '
                    f'max_rows={max_rows}: dataset config not found.'
            )
            continue

        trimmed_csv_path = (
                INFORMER_DATASETS_ROOT / dataset_name / f'{dataset_name}_max_rows_{max_rows}.csv'
        )
        if not trimmed_csv_path.is_file():
            LOGGER.info(
                    f'Visualization skipped for dataset="{dataset_name}", '
                    f'max_rows={max_rows}: trimmed CSV "{trimmed_csv_path}" not found.'
            )
            continue

        train_df, valid_df = load_train_valid_for_visualization(
                csv_path=trimmed_csv_path,
                max_rows=max_rows,
                train_ratio=dataset_cfg.train_ratio,
        )

        if TARGET_COLUMN not in train_df.columns:
            LOGGER.info(
                    f'Visualization skipped for dataset="{dataset_name}", '
                    f'max_rows={max_rows}: target column "{TARGET_COLUMN}" missing in train_df.'
            )
            continue

        if 'date' not in train_df.columns or 'date' not in valid_df.columns:
            LOGGER.info(
                    f'Visualization skipped for dataset="{dataset_name}", '
                    f'max_rows={max_rows}: "date" column missing in train or valid DataFrame.'
            )
            continue

        train_dates = pd.to_datetime(train_df['date']).to_numpy()

        plot_time_series_single(
                x_values=train_dates,
                y_values=train_df[TARGET_COLUMN].to_numpy(),
                title=(
                    f'Informer input train series '
                    f'(dataset={dataset_name}, max_rows={max_rows}, column={TARGET_COLUMN})'
                ),
                label='y_train',
        )
        LOGGER.info(
                f'Train series plot built for dataset="{dataset_name}", max_rows={max_rows}.'
        )

        predictions_csv_path = build_predictions_csv_path(
                metrics_root_dir=INFORMER_METRICS_ROOT,
                dataset_name=dataset_name,
                max_rows=max_rows,
        )
        if not predictions_csv_path.is_file():
            LOGGER.info(
                    f'Predictions CSV not found at "{predictions_csv_path}" for '
                    f'dataset="{dataset_name}", max_rows={max_rows}; skipping prediction plots.'
            )
            continue

        df_pred = pd.read_csv(predictions_csv_path)

        required_columns = {'split', 'date', 'y_true', 'y_pred'}
        if not required_columns.issubset(df_pred.columns):
            LOGGER.info(
                    f'Predictions CSV at "{predictions_csv_path}" does not contain required '
                    f'columns {required_columns}; skipping prediction plots.'
            )
            continue

        df_pred = df_pred.copy()
        df_pred['date'] = pd.to_datetime(df_pred['date'])

        train_pred_df = df_pred[df_pred['split'] == 'train']
        valid_pred_df = df_pred[df_pred['split'] == 'valid']

        if not train_pred_df.empty:
            train_pred_dates = train_pred_df['date'].to_numpy()
            train_y_true = train_pred_df['y_true'].to_numpy()
            train_y_pred = train_pred_df['y_pred'].to_numpy()

            plot_true_vs_pred(
                    x_values=train_pred_dates,
                    y_true=train_y_true,
                    y_pred=train_y_pred,
                    title=(
                        f'Informer+Optuna train predictions '
                        f'(dataset={dataset_name}, max_rows={max_rows})'
                    ),
            )
            LOGGER.info(
                    f'Train prediction plot built for dataset="{dataset_name}", max_rows={max_rows}.'
            )
        else:
            LOGGER.info(
                    f'Train predictions missing in CSV for dataset="{dataset_name}", '
                    f'max_rows={max_rows}.'
            )

        if not valid_pred_df.empty:
            valid_pred_dates = valid_pred_df['date'].to_numpy()
            valid_y_true = valid_pred_df['y_true'].to_numpy()
            valid_y_pred = valid_pred_df['y_pred'].to_numpy()

            plot_true_vs_pred(
                    x_values=valid_pred_dates,
                    y_true=valid_y_true,
                    y_pred=valid_y_pred,
                    title=(
                        f'Informer+Optuna validation predictions '
                        f'(dataset={dataset_name}, max_rows={max_rows})'
                    ),
            )
            LOGGER.info(
                    f'Validation prediction plot built for dataset="{dataset_name}", max_rows={max_rows}.'
            )
        else:
            LOGGER.info(
                    f'Validation predictions missing in CSV for dataset="{dataset_name}", '
                    f'max_rows={max_rows}.'
            )